In [1]:
import sys
import os

def get_UGCE_directory():
    """Get the path of the 'UGCE-User-Guided-Counterfactual-Exploration' directory."""
    current_dir = os.getcwd()
    target_dir = 'UGCE-User-Guided-Counterfactual-Exploration'
    
    while os.path.basename(current_dir) != target_dir:
        current_dir = os.path.dirname(current_dir)
        if current_dir == os.path.dirname(current_dir):
            return None
        
    return current_dir

def get_system_slash():
    """Get the system-specific directory separator."""
    return os.sep

UGCE_dir = get_UGCE_directory()
sys.path.append(UGCE_dir)
sep = get_system_slash()
sys.path.append(UGCE_dir + get_system_slash() + 'src')

from dataLoader import *
from utils import *
from test_utils import *

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
seed_number = 42
import random

random.seed(seed_number)
np.random.seed(seed_number)

In [4]:
datasetName = "GermanCredit"

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import pandas as pd
import dice_ml
from dice_ml.utils import helpers

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

names = ['existingchecking', 'duration', 'credithistory', 'purpose', 'creditamount', 
         'savings', 'employmentsince', 'installmentrate', 'statussex', 'otherdebtors', 
         'residencesince', 'property', 'age', 'otherinstallmentplans', 'housing', 
         'existingcredits', 'job', 'peopleliable', 'telephone', 'foreignworker', 'target']
dataset = pd.read_csv(f"{ugce_dir}/data/GermanCredit.data", sep=' ', header=None ,names = names)
TARGET_COLUMN = 'target'
dataset[TARGET_COLUMN] = LabelEncoder().fit_transform(dataset[TARGET_COLUMN])
datasetX = dataset.drop(columns=[TARGET_COLUMN])
target = dataset[TARGET_COLUMN]

x_train, x_test, y_train, y_test = train_test_split(datasetX,
                                                    target,
                                                    test_size=0.2,
                                                    random_state=0,
                                                    stratify=target)

numerical = datasetX.select_dtypes(include=[np.number]).columns.tolist()
categorical = x_train.columns.difference(numerical)

numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

transformations = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical),
        ('cat', categorical_transformer, categorical)])

model = RandomForestClassifier(random_state=42)

model = Pipeline(steps=[('preprocessor', transformations),
                      ('classifier', model)])

model.fit(x_train, y_train)

y_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

negative_instances = x_test[model.predict(x_test) == 0]
instances_to_explain = negative_instances
print("Number of instances to explain: ", len(instances_to_explain))

Accuracy:  0.75
Number of instances to explain:  166


In [6]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

In [7]:
numerical_columns = iea.dataset.select_dtypes(include=['int64', 'float64']).columns

non_zero_descriptions = {}

for col in numerical_columns:
    non_zero_values = iea.dataset[iea.dataset[col] != 0][col]
    if not non_zero_values.empty:
        non_zero_descriptions[col] = non_zero_values.describe()

# Display results
for feature, stats in non_zero_descriptions.items():
    print(f"\n Feature: {feature}")
    print(stats)


 Feature: existingchecking
count    726.000000
mean       2.172176
std        0.940637
min        1.000000
25%        1.000000
50%        3.000000
75%        3.000000
max        3.000000
Name: existingchecking, dtype: float64

 Feature: duration
count    1000.000000
mean       20.903000
std        12.058814
min         4.000000
25%        12.000000
50%        18.000000
75%        24.000000
max        72.000000
Name: duration, dtype: float64

 Feature: credithistory
count    960.000000
mean       2.651042
std        0.969880
min        1.000000
25%        2.000000
50%        2.000000
75%        4.000000
max        4.000000
Name: credithistory, dtype: float64

 Feature: purpose
count    766.000000
mean       4.278068
std        2.347512
min        1.000000
25%        3.000000
50%        4.000000
75%        4.000000
max        9.000000
Name: purpose, dtype: float64

 Feature: creditamount
count     1000.000000
mean      3271.258000
std       2822.736876
min        250.000000
25%       13

# UGCE

## Dynamic

# Only Immutability

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    'age': 'i',
}
for col in iea.feature_names:
    if col not in updated_constraints:
        updated_constraints[col] = ''
updated_constraints

results_incremental_explainer_immutability = []
for i in range(5):
    import time
    strategy = "fix_population_update_fitness"
    results_incremental = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.7, data_distribution=True,
        strategy="fix_population_update_fitness", population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=False,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_explainer_immutability.append(results_incremental)
import os
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_explainer_immutability, open(f'{results_dir}/results_incremental{strategy}_only_immutability_consts_ONE_constraint.pkl', 'wb'))

In [9]:
from test_utils import *

aggregate_results_incremental(iea, results_incremental_explainer_immutability, verbose=True)

Full Time: mean = 28.02, std = 0.00
Generations: mean = 6.00, std = 0.00
Coverage: mean = 92.86, std = 0.00
Proximity Loss: mean = 0.06, std = 0.00
Sparsity: mean = 0.02, std = 0.00
Intermediate Best Distances: mean = 0.02, std = 0.00


(28.018463850021362,
 6.0,
 92.85714285714286,
 0.06195217839649393,
 0.022612179487179477,
 0.01857387907271897)

# Only Range

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    'age': (34, 75)
}
for col in iea.feature_names:
    if col not in updated_constraints:
        updated_constraints[col] = ''
updated_constraints

results_incremental_explainer_ranges = []
for i in range(5):
    import time
    strategy = "fix_population_update_fitness"
    results_incremental = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.7, data_distribution=True,
        strategy="fix_population_update_fitness", population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=False,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_explainer_ranges.append(results_incremental)
import os
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_explainer_ranges, open(f'{results_dir}/results_incremental{strategy}_only_range_consts_ONE_constraint.pkl', 'wb'))

In [11]:
from test_utils import *

aggregate_results_incremental(iea, results_incremental_explainer_ranges, verbose=True)

Full Time: mean = 20.97, std = 0.00
Generations: mean = 6.00, std = 0.00
Coverage: mean = 64.88, std = 0.00
Proximity Loss: mean = 0.08, std = 0.00
Sparsity: mean = 0.02, std = 0.00
Intermediate Best Distances: mean = 0.04, std = 0.00


(20.96557879447937,
 6.0,
 64.88095238095238,
 0.07564195349695249,
 0.024220183486238517,
 0.03750789359784869)

# Only Directionality

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    "age": 'incr'
}
for col in iea.feature_names:
    if col not in updated_constraints:
        updated_constraints[col] = ''
updated_constraints


results_incremental_explainer_direct = []
for i in range(5):
    import time
    strategy = "fix_population_update_fitness"
    results_incremental = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.7, data_distribution=True,
        strategy="fix_population_update_fitness", population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=False,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_explainer_direct.append(results_incremental)
import os
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_explainer_direct, open(f'{results_dir}/results_incremental{strategy}_only_directionality_consts_ONE_constraint.pkl', 'wb'))

In [13]:
from test_utils import *

aggregate_results_incremental(iea, results_incremental_explainer_direct, verbose=True)

Full Time: mean = 16.45, std = 0.36
Generations: mean = 6.00, std = 0.00
Coverage: mean = 53.57, std = 0.00
Proximity Loss: mean = 0.07, std = 0.00
Sparsity: mean = 0.03, std = 0.00
Intermediate Best Distances: mean = 0.03, std = 0.00


(16.454408009847004,
 6.0,
 53.57142857142858,
 0.07017988599581793,
 0.025379629629629624,
 0.032405259087393654)